In [ ]:
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt

import sys
sys.path.append("../")
from datasets import ImageNet, Places365, ArtPlaces, INaturalist

### Konstanten

In [ ]:
# Visualization settings
NORMALIZE_CONFUSION_MATRIX = True
DIFFERENT_COLORS_FOR_DIAG = False
HIGHLIGHT_DIAG = False

RESULTS_TO_VISUALIZE = [
    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\ImageNet_linear_and_rest_20260609_104513\results.json",
        "name": "ConvNeXt_v2",
        "dataset": "ImageNet",
        "k": 5
    },
    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\ImageNet_linear_and_rest_20260609_104513\results.json",
        "name": "Linear VICReg",
        "dataset": "ImageNet",
        "k": 5
    },

    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\Places365_linear_and_rest_20260608_160620\results.json",
        "name": "DINO_v2",
        "dataset": "Places365",
        "k": 5
    },
    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\Places365_pooling_20260612_171445\results.json",
        "name": "Pooling VICReg",
        "dataset": "Places365",
        "k": 5
    },

    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\ArtPlaces_linear_and_rest_20260608_144803\results.json",
        "name": "DINO_v2",
        "dataset": "ArtPlaces",
        "k": 5
    },
    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\ArtPlaces_attention_20260615_211841\results.json",
        "name": "Attention MoCo",
        "dataset": "ArtPlaces",
        "k": 5
    },

    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\iNaturalist_linear_and_rest_20260609_133057\results.json",
        "name": "DINO_v2",
        "dataset": "iNaturalist_Family",
        "k": 5
    },
    {
        "path": r"D:\Dokumente\Studium\Masterprojekt\Evaluation\iNaturalist_linear_and_rest_20260609_133057\results.json",
        "name": "Linear VICReg",
        "dataset": "iNaturalist_Family",
        "k": 5
    },
]

### Fehlermatrix visualisieren

In [ ]:
def visualize_confusion_matrix(cm, method_name, dataset_name):
    # cm = np.array(data[NAME][DATASET]["per_class"]["matrix"][METRIC + "@" + str(K)])

    if NORMALIZE_CONFUSION_MATRIX:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = ".2f"
    else:
        fmt = "d"

    plt.figure(method_name + " - " + dataset_name, figsize=(8, 6))
    plt.rcParams.update({'font.size': 25})

    if DIFFERENT_COLORS_FOR_DIAG:
        # Masken
        diagonal_mask = np.eye(cm.shape[0], dtype=bool)
        off_diag_mask = ~diagonal_mask

        # Off-Diagonal Heatmap (Rot)
        sns.heatmap(cm, mask=diagonal_mask, annot=False, fmt=fmt, cmap="Reds", cbar=False)

        # Diagonal Heatmap (Grün) darüber plotten
        sns.heatmap(cm, mask=off_diag_mask, annot=False, fmt=fmt, cmap="Greens", cbar=False)
    else:
        sns.heatmap(cm, annot=False, fmt=fmt, cmap="Blues", cbar=False)

    if HIGHLIGHT_DIAG:
        # Diagonale markieren
        for i in range(cm.shape[0]):
            plt.gca().add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1))

    # plt.xlabel('Vorhergesagte Klasse')
    # plt.ylabel('Richtige Klasse')
    plt.xlabel('predicted class')
    plt.ylabel('true class')
    # plt.title(method_name + " - " + dataset_name)
    plt.show()

In [ ]:
%matplotlib qt

for result in RESULTS_TO_VISUALIZE:
    with open(result["path"], "r") as f:
        data = json.load(f)
    
    name = result["name"]
    dataset = result["dataset"]
    k = result["k"]

    cm = np.array(data[dataset][name]["matrix"][str(k)])
    visualize_confusion_matrix(cm, name, dataset)

### Größte Fehler ausgeben

In [ ]:
imagenet = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\ImageNet")
places365 = Places365(root=r"C:\Users\mariu\Documents\Development\Datasets\Places365")
artplaces = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280")
inaturalist = INaturalist(root=r"D:\Dokumente\Development\Datasets\iNaturalist", target_type="family", split="train_mini")

In [ ]:
def get_top_confusions(cm, n=5, class_labels=None):
    """
    Findet die n größten Verwechslungen in der Confusion-Matrix.
    
    cm: numpy array (Confusion Matrix)
    n: Anzahl der Top-Verwechslungen
    class_labels: Liste der Klassennamen (optional)
    
    Returns:
        list of tuples: [(true_class, predicted_class, count), ...]
    """
    
    cm_copy = cm.copy()
    # Diagonale ignorieren (korrekte Vorhersagen)
    np.fill_diagonal(cm_copy, -1)
    
    # Flatten, sortieren, die größten n auswählen
    flat_indices = cm_copy.flatten().argsort()[::-1][:n]
    top_confusions = []
    
    n_classes = cm.shape[0]
    if class_labels is None:
        class_labels = list(range(n_classes))
    
    for idx in flat_indices:
        i = idx // n_classes  # Zeile
        j = idx % n_classes   # Spalte
        top_confusions.append((class_labels[i], class_labels[j], cm[i, j]))
    
    return top_confusions


In [ ]:
for result in RESULTS_TO_VISUALIZE:
    with open(result["path"], "r") as f:
        data = json.load(f)
    
    name = result["name"]
    dataset = result["dataset"]
    k = result["k"]

    match dataset:
        case "ImageNet":
            d = imagenet
        case "Places365":
            d = places365
        case "ArtPlaces":
            d = artplaces
        case "iNaturalist_Family":
            d = inaturalist

    cm = np.array(data[dataset][name]["matrix"][str(k)])
    top_confusions = get_top_confusions(cm, n=5)
    print(dataset + " - " + name + ":")
    for c in top_confusions:
        r = d.idx_to_class[c[0]]
        p = d.idx_to_class[c[1]]
        print(f"Richtig: {r}, Vorhergesagt: {p}, Fehler: {c[2]}")
    print()